## Scraping from an example website

In [6]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

# Set up Chrome options to disable loading images, JavaScript, and CSS
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode (no GUI)
chrome_options.add_argument("--disable-gpu")  # Disable GPU (useful for headless mode)
chrome_options.add_argument("--disable-extensions")  # Disable extensions
chrome_options.add_argument("--disable-infobars")  # Disable infobars
chrome_options.add_argument("--disable-dev-shm-usage")  # Overcome limited resource problems
chrome_options.add_argument("--no-sandbox")  # Bypass OS security model

# Disable images, JavaScript, and CSS
chrome_prefs = {
    "profile.managed_default_content_settings.images": 2,  # Disable images
    "profile.managed_default_content_settings.stylesheets": 2,  # Disable CSS
    "profile.managed_default_content_settings.javascript": 2  # Disable JS
}
chrome_options.add_experimental_option("prefs", chrome_prefs)

driver = webdriver.Chrome(options=chrome_options)

url = 'https://books.toscrape.com/'
driver.get(url)

# Find all product
products = driver.find_elements(By.CLASS_NAME, 'product_pod')

# List untuk nampung data buku yang diambil
books_data = []

# Loop untuk setiap produk dan ekstrak informasinya
for product in products:
    # Title
    title = product.find_element(By.TAG_NAME, 'h3').find_element(By.TAG_NAME, 'a').get_attribute('title')
    
    # Price
    price = product.find_element(By.CLASS_NAME, 'price_color').text
    
    # Rating (info rating ada di class star-rating)
    rating_class = product.find_element(By.CLASS_NAME, 'star-rating').get_attribute('class')
    rating = rating_class.split()[-1] # Rating ada di class terakhir
    
    # Cek stock buku
    availability = product.find_element(By.CLASS_NAME, 'instock').text.strip()
    
    # Add to list
    books_data.append({
        'title': title,
        'price': price,
        'rating': rating,
        'availability': availability
    })

driver.quit()

books_df = pd.DataFrame(books_data)

# Save to CSV file
books_df.to_csv('books_data.csv', index=False)

books_df

,title,price,rating,availability
0,A Light in the Attic,£51.77,Three,In stock
1,Tipping the Velvet,£53.74,One,In stock
2,Soumission,£50.10,One,In stock
3,Sharp Objects,£47.82,Four,In stock
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock
5,The Requiem Red,£22.65,One,In stock
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,Four,In stock
7,The Coming Woman: A Novel Based on the Life of...,£17.93,Three,In stock
8,The Boys in the Boat: Nine Americans and Their...,£22.60,Four,In stock
9,The Black Maria,£52.15,One,In stock


## Scraping Yahoo Finance

In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

# Ticker symbols
ticker_symbols = ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']

# Set up Chrome options to disable loading images, JavaScript, and CSS
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode (no GUI)
chrome_options.add_argument("--disable-gpu")  # Disable GPU (useful for headless mode)
chrome_options.add_argument("--disable-extensions")  # Disable extensions
chrome_options.add_argument("--disable-infobars")  # Disable infobars
chrome_options.add_argument("--disable-dev-shm-usage")  # Overcome limited resource problems
chrome_options.add_argument("--no-sandbox")  # Bypass OS security model

# Disable images, JavaScript, and CSS
chrome_prefs = {
    "profile.managed_default_content_settings.images": 2,  # Disable images
    "profile.managed_default_content_settings.stylesheets": 2,  # Disable CSS
    "profile.managed_default_content_settings.javascript": 2  # Disable JS
}
chrome_options.add_experimental_option("prefs", chrome_prefs)

driver = webdriver.Chrome(options=chrome_options)

# List untuk menyimpan semua data saham
stock_data_list = []

# Loop setiap ticker symbols
for ticker_symbol in ticker_symbols:

    url = f'https://finance.yahoo.com/quote/{ticker_symbol}'
    driver.get(url)
    
    # Logika scraping: ekstrak harga saham, perubahan pasar, dan persentase perubahan
    stock_data = {}
    
    try:
        # Price
        stock_price_element = driver.find_element(By.CSS_SELECTOR, f'fin-streamer[data-symbol="{ticker_symbol}"][data-field="regularMarketPrice"]')
        stock_price = stock_price_element.text

        # Market change
        market_change_element = driver.find_element(By.CSS_SELECTOR, f'fin-streamer[data-symbol="{ticker_symbol}"][data-field="regularMarketChange"]')
        market_change = market_change_element.text
        
        # Market change percent
        percent_change_element = driver.find_element(By.CSS_SELECTOR, f'fin-streamer[data-symbol="{ticker_symbol}"][data-field="regularMarketChangePercent"]')
        percent_change = percent_change_element.text

        # Post market change
        post_market_change_element = driver.find_element(By.CSS_SELECTOR, f'fin-streamer[data-symbol="{ticker_symbol}"][data-field="postMarketChange"]')
        post_market_change = post_market_change_element.text

        # Post market change percent
        post_percent_change_element = driver.find_element(By.CSS_SELECTOR, f'fin-streamer[data-symbol="{ticker_symbol}"][data-field="postMarketChangePercent"]')
        post_percent_change = post_percent_change_element.text
        
        # Add data ke dictionary
        stock_data['ticker'] = ticker_symbol
        stock_data['price'] = stock_price
        stock_data['market_change'] = market_change
        stock_data['percent_change'] = percent_change
        stock_data['post_market_change'] = post_market_change
        stock_data['post_percent_change'] = post_percent_change

    except Exception as e:
        print(f"Error saat scraping {ticker_symbol}: {e}")
        stock_data['ticker'] = ticker_symbol
        stock_data['price'] = None # Jika ada error, simpan None
        stock_data['market_change'] = None
        stock_data['percent_change'] = None
        stock_data['post_market_change'] = None
        stock_data['post_percent_change'] = None
    
    # Add to list
    stock_data_list.append(stock_data)

driver.quit()

stock_df = pd.DataFrame(stock_data_list)

# Save to CSV file
stock_df.to_csv('mag7_stocks_data.csv', index=False)

stock_df

,ticker,price,market_change,percent_change,post_market_change,post_percent_change
0,AAPL,231.41,+0.84,(+0.36%),-0.43,(-0.19%)
1,AMZN,187.83,+1.45,(+0.78%),-0.25,(-0.13%)
2,GOOGL,165.27,+2.55,(+1.57%),0.00,(0.00%)
3,META,573.25,+5.47,(+0.96%),-1.23,(-0.22%)
4,MSFT,428.15,+3.42,(+0.81%),-0.64,(-0.15%)
5,NVDA,141.54,+1.13,(+0.80%),-0.46,(-0.32%)
6,TSLA,269.19,+8.71,(+3.34%),-2.11,(-0.78%)
